# C01 — Causal Inference I: DiD, RDD & Potential Outcomes

## Why causal inference matters for data scientists

Correlation is not causation — but most ML models only capture correlation. Causal inference answers a different question: **what would have happened if we had made a different decision?** This is the language of business: "Did our new checkout flow cause more conversions?" Not just "are users who used the new flow more likely to convert?"

## The Potential Outcomes Framework (Rubin Causal Model)

For each unit $i$, define two **potential outcomes**:
- $Y_i(1)$: outcome if unit $i$ receives treatment
- $Y_i(0)$: outcome if unit $i$ does not receive treatment

The **individual treatment effect** is: $\tau_i = Y_i(1) - Y_i(0)$

**The fundamental problem of causal inference:** We can only ever observe one of these — the factual. The other is counterfactual and unobservable.

We can only estimate *average* treatment effects:
- **ATE** (Average Treatment Effect): $\mathbb{E}[Y(1) - Y(0)]$
- **ATT** (Average Treatment Effect on the Treated): $\mathbb{E}[Y(1) - Y(0) | T=1]$

**Reference:** [econml docs](https://econml.azurewebsites.net/) | [statsmodels docs](https://www.statsmodels.org/stable/index.html)


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.linear_model import OLS
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Synthetic DID dataset ──────────────────────────────────────────────────────
# Scenario: A retailer launches a loyalty program in some stores (treatment)
# in Q3 2022. We observe Q2 (pre) and Q4 (post) revenue per store.
n_stores = 400
store_id = np.arange(n_stores)
treated = (store_id >= 200).astype(int)  # stores 200-399 are treated

# Underlying quality of store (confounder — better stores more likely treated)
store_quality = np.random.normal(0, 1, n_stores)
treated_biased = (store_quality + np.random.normal(0, 0.5, n_stores) > 0).astype(int)

# True treatment effect = 15% revenue increase
TRUE_EFFECT = 0.15

# Pre-period revenue (Q2 2022)
revenue_pre = 100_000 * np.exp(0.5 * store_quality + np.random.normal(0, 0.1, n_stores))

# Post-period revenue (Q4 2022)
time_trend = 0.03  # 3% natural growth
revenue_post = revenue_pre * (1 + time_trend) * np.exp(
    TRUE_EFFECT * treated + np.random.normal(0, 0.1, n_stores)
)

did_df = pd.DataFrame({
    'store_id': np.tile(store_id, 2),
    'treated': np.tile(treated, 2),
    'post': np.repeat([0, 1], n_stores),
    'revenue': np.concatenate([revenue_pre, revenue_post]),
    'store_quality': np.tile(store_quality, 2)
})
did_df['log_revenue'] = np.log(did_df['revenue'])
did_df['treated_post'] = did_df['treated'] * did_df['post']

# ── Synthetic RDD dataset ──────────────────────────────────────────────────────
# Scenario: Students scoring >= 70 on an exam receive tutoring.
# Does tutoring improve final exam performance?
n_students = 2000
exam_score = np.random.normal(65, 15, n_students)  # running variable
cutoff = 70
above_cutoff = (exam_score >= cutoff).astype(int)
TRUE_RDD_EFFECT = 8  # tutoring improves final score by 8 points

final_score = (
    40
    + 0.6 * exam_score               # relationship with running variable
    + TRUE_RDD_EFFECT * above_cutoff  # treatment effect at cutoff
    + np.random.normal(0, 5, n_students)
)

rdd_df = pd.DataFrame({
    'exam_score': exam_score,
    'above_cutoff': above_cutoff,
    'final_score': final_score,
    'centered_score': exam_score - cutoff  # running variable centered at cutoff
})

print(f"DiD dataset: {did_df.shape} | RDD dataset: {rdd_df.shape}")
print(f"True DiD effect: {TRUE_EFFECT:.0%} revenue increase")
print(f"True RDD effect: {TRUE_RDD_EFFECT} points")

---
## Exercise 1 — Naive Estimator: Why Correlation Fails

**Task:** Show why a naive comparison between treated and control groups gives the wrong answer — then explain why.

1. Compute the **naive ATT**: mean revenue of treated stores in post-period minus mean revenue of control stores in post-period.
2. Compare to the **true effect** (15%).
3. Run a simple OLS regression: `log_revenue ~ treated` on post-period only.
4. Explain in markdown: why is the naive estimate biased? What is the confounding mechanism in this dataset?
5. Return `naive_comparison`: dict with `naive_pct_effect`, `ols_coef`, `bias_direction`.

In [ ]:
# YOUR CODE HERE
naive_comparison = None

In [ ]:
# --- ASSERTIONS ---
assert naive_comparison is not None
assert set(naive_comparison.keys()) == {'naive_pct_effect', 'ols_coef', 'bias_direction'}
assert naive_comparison['bias_direction'] in ('upward', 'downward')
# Naive must be biased (not equal to true effect)
assert abs(naive_comparison['naive_pct_effect'] - TRUE_EFFECT) > 0.01, \
    "Naive estimate should be biased away from true effect"
print(f"✓ Exercise 1 passed")
print(f"True effect: {TRUE_EFFECT:.2%} | Naive estimate: {naive_comparison['naive_pct_effect']:.2%}")
print(f"Bias direction: {naive_comparison['bias_direction']}")

**Why the naive estimator is biased:** *(Explain the confounding mechanism. What assumption does the naive comparison violate?)*

---
## Exercise 2 — Difference-in-Differences (DiD)

## The Math

DiD compares the **change** in outcomes over time between treated and control groups:
$$\hat{\tau}_{\text{DiD}} = (\bar{Y}_{T,\text{post}} - \bar{Y}_{T,\text{pre}}) - (\bar{Y}_{C,\text{post}} - \bar{Y}_{C,\text{pre}})$$

**Key assumption (parallel trends):** In the absence of treatment, treated and control units would have followed the same trend over time.

**Regression form:** $Y_{it} = \alpha + \beta_1 \text{Treated}_i + \beta_2 \text{Post}_t + \beta_3 (\text{Treated}_i \times \text{Post}_t) + \epsilon_{it}$

The coefficient $\beta_3$ is the DiD estimate of the treatment effect.

**Task:**
1. Compute DiD manually using the 4-cell means formula.
2. Run the regression formulation using statsmodels OLS.
3. Extract the DiD coefficient, its standard error, t-stat, p-value, and 95% CI.
4. Compute `pct_effect` = $\exp(\hat{\beta}_3) - 1$ (since we're using log revenue).
5. Return `did_result`: dict with all metrics.

In [ ]:
# YOUR CODE HERE
did_result = None

In [ ]:
# --- ASSERTIONS ---
assert did_result is not None
required = ['did_coef', 'std_err', 't_stat', 'p_value', 'ci_lower', 'ci_upper', 'pct_effect']
for k in required:
    assert k in did_result, f"Missing: {k}"

# DiD estimate should be close to true effect (15%)
assert abs(did_result['pct_effect'] - TRUE_EFFECT) < 0.05, \
    f"DiD estimate {did_result['pct_effect']:.2%} too far from true {TRUE_EFFECT:.2%}"

# Should be statistically significant
assert did_result['p_value'] < 0.05, "DiD should be significant"

# CI must contain true effect
assert did_result['ci_lower'] < np.log(1 + TRUE_EFFECT) < did_result['ci_upper'], \
    "True effect must be in confidence interval"

print(f"✓ Exercise 2 passed")
print(f"DiD estimate: {did_result['pct_effect']:.2%} (true: {TRUE_EFFECT:.2%})")
print(f"p-value: {did_result['p_value']:.4f}")
print(f"95% CI: [{np.exp(did_result['ci_lower'])-1:.2%}, {np.exp(did_result['ci_upper'])-1:.2%}]")

---
## Exercise 3 — Parallel Trends Test

**The parallel trends assumption cannot be proven — only falsified.** The standard test: add pre-treatment periods and check that the treatment coefficient is near zero before treatment.

1. Create a synthetic 3-period dataset: `t=-1` (pre-pre), `t=0` (pre), `t=1` (post) using `did_df`.
   - For `t=-1`: simulate revenue as `revenue_pre * 0.95` (5% earlier)
2. Run the event study regression: include period dummies and interaction terms `treated × period_dummy` for each period. Omit `t=0` (pre-period) as the reference.
3. The coefficient on `treated × (t=-1)` is the **pre-trend test**: if it's statistically significant, parallel trends is violated.
4. Return `parallel_trends_result`: dict with pre-trend coefficient, p-value, and `parallel_trends_holds` (bool).

In [ ]:
# Create 3-period dataset
pre_pre = did_df[did_df['post'] == 0].copy()
pre_pre['post'] = -1
pre_pre['revenue'] = pre_pre['revenue'] * 0.95
pre_pre['log_revenue'] = np.log(pre_pre['revenue'])
pre_pre['treated_post'] = 0

three_period = pd.concat([pre_pre, did_df], ignore_index=True)

# YOUR CODE HERE
parallel_trends_result = None

In [ ]:
# --- ASSERTIONS ---
assert parallel_trends_result is not None
assert 'pre_trend_coef' in parallel_trends_result
assert 'pre_trend_pvalue' in parallel_trends_result
assert 'parallel_trends_holds' in parallel_trends_result
assert isinstance(parallel_trends_result['parallel_trends_holds'], bool)
# Since we generated data with parallel trends by design, it should hold
assert parallel_trends_result['parallel_trends_holds'], \
    "Parallel trends should hold in this synthetic dataset"
print(f"✓ Exercise 3 passed")
print(f"Pre-trend coef: {parallel_trends_result['pre_trend_coef']:.4f}")
print(f"Pre-trend p-value: {parallel_trends_result['pre_trend_pvalue']:.4f}")
print(f"Parallel trends holds: {parallel_trends_result['parallel_trends_holds']}")

---
## Exercise 4 — Regression Discontinuity Design (RDD)

## The Math

RDD exploits a sharp cutoff in treatment assignment. Units just above and just below the cutoff are *as good as randomly assigned* — they differ only in whether they crossed the threshold.

**Sharp RDD estimator:** fit separate regression lines on each side of the cutoff. The treatment effect is the **discontinuity** (jump) at the cutoff:
$$\hat{\tau}_{\text{RDD}} = \lim_{x \downarrow c} E[Y|X=x] - \lim_{x \uparrow c} E[Y|X=x]$$

**Regression form:** Fit:
$$Y_i = \alpha + \beta_1 D_i + \beta_2 (X_i - c) + \beta_3 D_i(X_i - c) + \epsilon_i$$

where $D_i = \mathbf{1}[X_i \geq c]$ and $\beta_1$ is the RDD estimate.

**Bandwidth selection:** Only use observations within a window $[c-h, c+h]$ around the cutoff.

**Task:**
1. Implement the RDD regression with interaction term.
2. Test three bandwidths: h = 5, 10, 20. Report estimate and SE for each.
3. Return `rdd_results`: DataFrame with bandwidth, estimate, std_err, p_value, n_obs.

In [ ]:
def estimate_rdd(df: pd.DataFrame, running_var: str, outcome: str,
                  cutoff: float, bandwidth: float) -> dict:
    """
    Sharp RDD estimator using local linear regression.
    Returns dict: estimate, std_err, t_stat, p_value, ci_lower, ci_upper, n_obs
    """
    # YOUR CODE HERE
    # 1. Filter to bandwidth window
    # 2. Create D = 1(running_var >= cutoff) and centered running var
    # 3. Create interaction: D * centered_running_var
    # 4. OLS: outcome ~ D + centered_running_var + D*centered_running_var
    # 5. Extract D coefficient = RDD estimate
    pass

bandwidths = [5, 10, 20]
rdd_results = pd.DataFrame([
    estimate_rdd(rdd_df, 'exam_score', 'final_score', cutoff, bw)
    for bw in bandwidths
])
rdd_results.insert(0, 'bandwidth', bandwidths)
rdd_results

In [ ]:
# --- ASSERTIONS ---
assert len(rdd_results) == 3
for col in ['estimate', 'std_err', 'p_value', 'n_obs']:
    assert col in rdd_results.columns, f"Missing: {col}"

# All estimates should be close to true effect (8 points)
for _, row in rdd_results.iterrows():
    assert abs(row['estimate'] - TRUE_RDD_EFFECT) < 3, \
        f"RDD estimate {row['estimate']:.2f} too far from true {TRUE_RDD_EFFECT}"
    assert row['p_value'] < 0.05, f"RDD should be significant at bw={row['bandwidth']}"

print(f"✓ Exercise 4 passed")
print(rdd_results.to_string(index=False))

---
## Exercise 5 — RDD Validity Tests

**RDD has two testable assumptions:**

1. **No manipulation of the running variable:** Units shouldn't be able to precisely control whether they're above/below the cutoff. Test: the density of the running variable should be smooth through the cutoff (McCrary density test).

2. **Continuity of covariates at the cutoff:** Pre-treatment covariates should not jump at the cutoff — only the outcome should.

**Task:**
1. Implement a simple density continuity test: fit local linear regressions to the histogram density on each side of the cutoff. Is there a discontinuity in the count of observations?
2. Run placebo RDD: use a fake cutoff (e.g., at 60) where there is no treatment. Verify the "effect" is near zero.
3. Return `rdd_validity`: dict with `density_test_pvalue`, `placebo_estimate`, `placebo_pvalue`.

In [ ]:
# YOUR CODE HERE
rdd_validity = None

In [ ]:
# --- ASSERTIONS ---
assert rdd_validity is not None
assert 'density_test_pvalue' in rdd_validity
assert 'placebo_estimate' in rdd_validity
assert 'placebo_pvalue' in rdd_validity
# Placebo should NOT be significant (no real effect at fake cutoff)
assert rdd_validity['placebo_pvalue'] > 0.05, \
    f"Placebo effect should be insignificant, got p={rdd_validity['placebo_pvalue']:.4f}"
assert abs(rdd_validity['placebo_estimate']) < 5, "Placebo estimate should be near zero"
print(f"✓ Exercise 5 passed")
print(f"Density test p-value: {rdd_validity['density_test_pvalue']:.4f}")
print(f"Placebo estimate: {rdd_validity['placebo_estimate']:.3f} (p={rdd_validity['placebo_pvalue']:.4f})")

---
## Exercise 6 — Instrumental Variables (IV)

## The Math

When treatment is endogenous (correlated with the error term), OLS is biased. IV uses an **instrument** $Z$ that:
1. Affects treatment $D$ (relevance)
2. Only affects outcome $Y$ through $D$ (exclusion restriction)
3. Is independent of confounders (exogeneity)

**Two-Stage Least Squares (2SLS):**
- Stage 1: Regress $D$ on $Z$ (and controls), get predicted values $\hat{D}$
- Stage 2: Regress $Y$ on $\hat{D}$ (and controls)

The 2SLS estimate identifies the **LATE** (Local Average Treatment Effect) — the effect for **compliers** (units who take treatment when assigned, and don't when not assigned).

**Task:** Implement 2SLS manually. Scenario: does education (D) increase wages (Y)? Use proximity to college as instrument (Z).

In [ ]:
# Synthetic IV dataset: education → wages, with ability as confounder
np.random.seed(99)
n = 2000
ability = np.random.normal(0, 1, n)        # unobserved confounder
near_college = np.random.binomial(1, 0.5, n)  # instrument: near college (random)

# Stage 1: education depends on instrument + ability
education = (12 + 2 * near_college + 1.5 * ability
             + np.random.normal(0, 1, n)).clip(8, 20)

# Outcome: wages depend on education + ability (confounded!)
TRUE_IV_EFFECT = 0.08  # 8% wage increase per year of education
log_wage = (1.5 + TRUE_IV_EFFECT * education + 0.3 * ability
            + np.random.normal(0, 0.2, n))

iv_df = pd.DataFrame({
    'log_wage': log_wage,
    'education': education,
    'near_college': near_college,
    'ability': ability  # unobserved in practice — only for verification
})

def two_stage_least_squares(df: pd.DataFrame, outcome: str, treatment: str,
                              instrument: str, controls: list = None) -> dict:
    """
    2SLS estimator.
    Stage 1: treatment ~ instrument [+ controls]
    Stage 2: outcome ~ treatment_hat [+ controls]
    Returns: stage1_fstat, iv_estimate, iv_se, iv_pvalue, ols_estimate (for comparison)
    """
    # YOUR CODE HERE
    pass

iv_result = two_stage_least_squares(iv_df, 'log_wage', 'education', 'near_college')

In [ ]:
# --- ASSERTIONS ---
assert iv_result is not None
required = ['stage1_fstat', 'iv_estimate', 'iv_se', 'iv_pvalue', 'ols_estimate']
for k in required:
    assert k in iv_result, f"Missing: {k}"

# IV should be closer to true effect than OLS
iv_error = abs(iv_result['iv_estimate'] - TRUE_IV_EFFECT)
ols_error = abs(iv_result['ols_estimate'] - TRUE_IV_EFFECT)
print(f"True effect: {TRUE_IV_EFFECT:.4f}")
print(f"IV estimate: {iv_result['iv_estimate']:.4f} (error: {iv_error:.4f})")
print(f"OLS estimate: {iv_result['ols_estimate']:.4f} (error: {ols_error:.4f})")

# Stage 1 F-stat should be > 10 (weak instrument test)
assert iv_result['stage1_fstat'] > 10, \
    f"Weak instrument: F={iv_result['stage1_fstat']:.1f} < 10"

assert abs(iv_result['iv_estimate'] - TRUE_IV_EFFECT) < 0.04
print(f"✓ Exercise 6 passed — Stage 1 F-stat: {iv_result['stage1_fstat']:.1f}")

---
## Exercise 7 — Propensity Score Matching (manual)

## The Math

**Propensity score:** $e(x) = P(T=1|X=x)$ — the probability of receiving treatment given observed covariates.

**Theorem (Rosenbaum & Rubin):** If $Y(0), Y(1) \perp T | X$ (no unobserved confounders), then $Y(0), Y(1) \perp T | e(X)$. You can condition on a scalar instead of the full covariate vector.

**Matching estimator:**
1. Estimate propensity scores (logistic regression)
2. For each treated unit, find the most similar control unit (nearest neighbor in propensity score)
3. ATT = mean outcome difference between matched pairs

**Task:** Implement PSM from scratch using the biased treatment assignment (`treated_biased`).

In [ ]:
from sklearn.linear_model import LogisticRegression

# Build dataset with confounded treatment assignment
psm_df = did_df[did_df['post'] == 1].copy()
psm_df['treated_biased'] = np.tile(treated_biased, 1)[:len(psm_df)]
psm_df['treated_biased'] = (psm_df['store_quality'] >
                              np.median(psm_df['store_quality'])).astype(int)

def propensity_score_matching(df: pd.DataFrame, treatment: str,
                               outcome: str, covariates: list,
                               caliper: float = 0.05) -> dict:
    """
    Propensity Score Matching (1:1 nearest neighbor with caliper).
    Returns: pscore_auc, n_matched, att_estimate, att_se, balance_before, balance_after
    """
    # YOUR CODE HERE
    # 1. Fit logistic regression to estimate propensity scores
    # 2. For each treated unit: find nearest control within caliper distance
    # 3. Compute ATT on matched sample
    # 4. Check covariate balance before/after (standardized mean difference)
    pass

psm_result = propensity_score_matching(
    psm_df, 'treated_biased', 'log_revenue', ['store_quality']
)

In [ ]:
# --- ASSERTIONS ---
assert psm_result is not None
required = ['pscore_auc', 'n_matched', 'att_estimate', 'att_se', 'balance_before', 'balance_after']
for k in required:
    assert k in psm_result, f"Missing: {k}"
assert psm_result['pscore_auc'] > 0.6, "Propensity model should have AUC > 0.6"
assert psm_result['n_matched'] > 0
# Balance should improve after matching (smaller SMD)
assert psm_result['balance_after'] <= psm_result['balance_before'], \
    "Matching must improve covariate balance"
print(f"✓ Exercise 7 passed")
print(f"Matched pairs: {psm_result['n_matched']}")
print(f"ATT estimate: {psm_result['att_estimate']:.4f}")
print(f"Balance (SMD) before: {psm_result['balance_before']:.4f} → after: {psm_result['balance_after']:.4f}")

---
## Exercise 8 — Synthetic Control Method

## The Math

When there is only **one treated unit** (e.g., one country, one city), DiD and matching don't apply. The **synthetic control** method builds a counterfactual by finding a weighted combination of control units that best matches the treated unit in the pre-treatment period.

$$\hat{Y}_{1t}(0) = \sum_{j=2}^{J+1} w_j^* Y_{jt} \quad \text{for } t > T_0$$

Weights $w^*$ are chosen to minimize pre-treatment fit:
$$w^* = \arg\min_{w \geq 0, \sum w = 1} \sum_{t \leq T_0} (Y_{1t} - \sum_j w_j Y_{jt})^2$$

**Task:** Implement synthetic control for a scenario where one store (store 0) receives a special intervention. Find optimal weights using scipy.optimize, construct the synthetic counterfactual, and estimate the treatment effect.

In [ ]:
from scipy.optimize import minimize

# Create time-series panel: 10 stores, 20 time periods, store 0 treated at t=10
np.random.seed(77)
n_units = 15
n_periods = 24
T0 = 12  # treatment starts at period 12
TRUE_SC_EFFECT = 10.0

unit_fe = np.random.normal(0, 5, n_units)  # unit fixed effects
time_fe = np.cumsum(np.random.normal(0.5, 0.3, n_periods))  # time trend
panel = pd.DataFrame({
    'unit': np.repeat(range(n_units), n_periods),
    'period': np.tile(range(n_periods), n_units),
    'y': (unit_fe.repeat(n_periods) + np.tile(time_fe, n_units)
          + np.random.normal(0, 1, n_units * n_periods))
})

# Add treatment effect to unit 0 in post periods
panel.loc[(panel['unit'] == 0) & (panel['period'] >= T0), 'y'] += TRUE_SC_EFFECT

def synthetic_control(panel: pd.DataFrame, treated_unit: int,
                       T0: int) -> dict:
    """
    Fit synthetic control weights.
    Returns: weights (array), synthetic_counterfactual (Series), att_estimate
    """
    # YOUR CODE HERE
    # 1. Pivot panel to wide format: periods x units
    # 2. Use scipy.optimize.minimize to find weights minimizing pre-period fit
    #    - Constraints: w >= 0, sum(w) = 1
    # 3. Synthetic = Y_controls @ w_optimal
    # 4. ATT = mean(Y_treated - synthetic) in post-period
    pass

sc_result = synthetic_control(panel, treated_unit=0, T0=T0)

In [ ]:
# --- ASSERTIONS ---
assert sc_result is not None
assert 'weights' in sc_result
assert 'att_estimate' in sc_result

weights = sc_result['weights']
assert abs(weights.sum() - 1.0) < 1e-6, "Weights must sum to 1"
assert (weights >= -1e-6).all(), "Weights must be non-negative"

# ATT should be close to true effect
assert abs(sc_result['att_estimate'] - TRUE_SC_EFFECT) < 5, \
    f"SC ATT {sc_result['att_estimate']:.2f} far from true {TRUE_SC_EFFECT}"

print(f"✓ Exercise 8 passed")
print(f"True effect: {TRUE_SC_EFFECT:.1f} | SC estimate: {sc_result['att_estimate']:.2f}")
print(f"Top donor weights: {sorted(zip(range(1,n_units), weights), key=lambda x: -x[1])[:3]}")

---
## Exercise 9 — Double-Robust Estimation

## The Math

The **Augmented IPW (AIPW)** estimator is doubly robust: it gives a consistent estimate of the ATE if *either* the outcome model OR the propensity model is correctly specified (but not necessarily both).

$$\hat{\tau}_{\text{AIPW}} = \frac{1}{n}\sum_i \left[\underbrace{\hat{\mu}_1(X_i) - \hat{\mu}_0(X_i)}_{\text{outcome model}} + \underbrace{\frac{T_i(Y_i - \hat{\mu}_1(X_i))}{\hat{e}(X_i)}}_{\text{treated residual}} - \underbrace{\frac{(1-T_i)(Y_i - \hat{\mu}_0(X_i))}{1-\hat{e}(X_i)}}_{\text{control residual}}\right]$$

where $\hat{\mu}_1, \hat{\mu}_0$ are outcome models for treated/control and $\hat{e}$ is the propensity score.

**Task:** Implement AIPW from scratch. Compare to naive OLS and IPW-only.

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import cross_val_predict

def aipw_estimator(X: np.ndarray, T: np.ndarray, Y: np.ndarray,
                    outcome_model=None, propensity_model=None,
                    n_folds: int = 5) -> dict:
    """
    Doubly-robust AIPW estimator.
    Uses cross-fitting (k-fold) to avoid overfitting.
    Returns: ate_estimate, ate_se, ate_ci, iptw_estimate, naive_estimate
    """
    # YOUR CODE HERE
    # 1. Estimate propensity scores e(X) using cross_val_predict
    # 2. Estimate mu_1(X) and mu_0(X) separately using outcome models
    # 3. Apply AIPW formula
    # 4. Bootstrap SE (200 iterations)
    pass

# Apply to confounded store dataset
post_data = psm_df.copy()
X_aipw = post_data[['store_quality']].values
T_aipw = post_data['treated_biased'].values
Y_aipw = post_data['log_revenue'].values

aipw_result = aipw_estimator(X_aipw, T_aipw, Y_aipw)

In [ ]:
# --- ASSERTIONS ---
assert aipw_result is not None
for k in ['ate_estimate', 'ate_se', 'iptw_estimate', 'naive_estimate']:
    assert k in aipw_result, f"Missing: {k}"
print(f"✓ Exercise 9 passed")
print(f"Naive OLS:    {aipw_result['naive_estimate']:.4f}")
print(f"IPTW:         {aipw_result['iptw_estimate']:.4f}")
print(f"AIPW (DR):    {aipw_result['ate_estimate']:.4f} ± {aipw_result['ate_se']:.4f}")

---
## Exercise 10 — Capstone: Full Causal Analysis Pipeline

**Spec:** A company launched a new pricing strategy in 40 of its 100 markets. You have 18 months of pre-launch data and 6 months post-launch.

Build a complete causal analysis:
1. **Exploratory analysis**: plot pre-trends for treated vs control markets.
2. **Parallel trends test**: formalize the pre-trend test.
3. **DiD estimate**: with market and time fixed effects.
4. **Robustness check**: use synthetic control as an alternative estimator.
5. **Heterogeneous effects**: split by market_size (large/small) — is the treatment effect different?
6. **Confidence intervals**: bootstrap the DiD estimate (1000 iterations).
7. Return `causal_report`: dict with all estimates, CIs, and a `conclusion` string.

In [ ]:
# Generate synthetic panel
np.random.seed(55)
n_markets = 100
n_months = 24  # 18 pre + 6 post
T0_month = 18
TRUE_CAPSTONE_EFFECT = 0.10  # 10% revenue increase

treated_markets = np.random.choice(n_markets, 40, replace=False)
market_size = np.random.choice(['large', 'small'], n_markets)
market_quality = np.random.normal(0, 1, n_markets)

rows = []
for m in range(n_markets):
    is_treated = m in treated_markets
    for t in range(n_months):
        is_post = t >= T0_month
        rev = (50000 * (1.2 if market_size[m]=='large' else 0.8)
               * np.exp(0.3 * market_quality[m] + 0.005 * t
                        + TRUE_CAPSTONE_EFFECT * is_treated * is_post
                        + np.random.normal(0, 0.05)))
        rows.append({'market': m, 'month': t, 'revenue': rev,
                     'treated': int(is_treated), 'post': int(is_post),
                     'market_size': market_size[m]})

capstone_panel = pd.DataFrame(rows)
capstone_panel['log_revenue'] = np.log(capstone_panel['revenue'])
capstone_panel['treated_post'] = capstone_panel['treated'] * capstone_panel['post']

# YOUR CODE HERE
causal_report = None

In [ ]:
# --- ASSERTIONS ---
assert causal_report is not None
required = ['did_estimate', 'did_pct_effect', 'bootstrap_ci', 'parallel_trends_pvalue',
            'hte_large', 'hte_small', 'conclusion']
for k in required:
    assert k in causal_report, f"Missing: {k}"

# DiD should recover true effect
assert abs(causal_report['did_pct_effect'] - TRUE_CAPSTONE_EFFECT) < 0.05

# Bootstrap CI must be a (lower, upper) tuple
ci = causal_report['bootstrap_ci']
assert len(ci) == 2 and ci[0] < ci[1]

# True effect must be in CI
assert ci[0] < TRUE_CAPSTONE_EFFECT < ci[1], "True effect must be in bootstrap CI"

assert isinstance(causal_report['conclusion'], str) and len(causal_report['conclusion']) > 20

print(f"✓ Exercise 10 passed — Full causal pipeline complete")
print(f"DiD effect: {causal_report['did_pct_effect']:.2%} (true: {TRUE_CAPSTONE_EFFECT:.2%})")
print(f"Bootstrap 95% CI: [{ci[0]:.2%}, {ci[1]:.2%}]")
print(f"Parallel trends p-value: {causal_report['parallel_trends_pvalue']:.4f}")
print(f"HTE — Large: {causal_report['hte_large']:.2%} | Small: {causal_report['hte_small']:.2%}")
print(f"Conclusion: {causal_report['conclusion']}")